## 03 - Data Cleaning

**Goal:** Clean the Online Retail II data using the problems documented in `02_data_quality_checks`, and save the result to `retail_clean.csv`.

**Raw data:** `retail_raw.csv` is never modified. All cleaning happens on a copy in memory.

**Cleaning plan:**
1. Load the raw data and fix data types
2. Drop the original `Sales` column
3. Remove the sheet-overlap duplicates
4. Separate returns, non-product codes, adjustments, and zero or negative price rows
5. Standardise `Description` and `Country`
6. Recalculate `Sales` as Quantity x Price
7. Save the clean data to `/clean/`

In [1]:
import pandas as pd

#### Step 1: Load the raw data and fix data types

In [2]:
df = pd.read_csv('retail_raw.csv')

In [3]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Sales
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


In [4]:
df.dtypes

Invoice         object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
Price          float64
Customer ID    float64
Country         object
Sales          float64
dtype: object

In [5]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [6]:
print(df['Invoice'].map(type).value_counts())
print(df['StockCode'].map(type).value_counts())

Invoice
<class 'str'>    1067371
Name: count, dtype: int64
StockCode
<class 'str'>    1067371
Name: count, dtype: int64


#### Step 2: Drop the original Sales column

In [7]:
df = df.drop(columns =['Sales'])

In [8]:
df.columns

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country'],
      dtype='object')

In [9]:
df.shape

(1067371, 8)

#### Step 3: Duplicates

In [10]:
overlap_end = df.iloc[:525461]['InvoiceDate'].max()

mask = (df.index >= 525461) & (df['InvoiceDate'] <= overlap_end)
print(mask.sum())

df = df[~mask]
print(df.shape)
print(df.duplicated().sum())

22523
(1044848, 8)
11812


In [11]:
dup = df[df.duplicated()]

In [12]:
dup_rev = (dup['Quantity']*dup['Price']).sum()

In [13]:
total_rev =(df['Quantity']*df['Price']).sum()

In [14]:
print(dup_rev)
print(total_rev)
print(dup_rev / total_rev * 100)

54228.42
18909762.118
0.2867747339263488


In [15]:
print(dup['Customer ID'].isnull().mean() * 100)
print(dup['Invoice'].nunique())

1.1513714866237725
4387


- **Removed:** 22,523 rows from the second sheet that repeat the overlap week (1 to 9 Dec 2010). These rows are identical in both sheets, so the first sheet's copy was kept. Shape went from 1,067,371 to 1,044,848 rows.
- **Kept:** 11,812 exact duplicates that occur within a single sheet. They make up 0.29% of revenue and sit on 4,387 invoices, and only 1.15% of them lack a Customer ID, so they appear mainly on customer invoices where the same item can plausibly repeat. The data cannot confirm whether they are errors.
- **Assumption:** Keeping them changes total revenue by less than 0.3% compared with dropping them.

#### Step 4: Row classification

In [16]:
df['row_type'] = 'sale'

groups = {
    'charges_fees': ['POST', 'DOT', 'C2', 'D', 'BANK CHARGES', 'AMAZONFEE', 'CRUK'],
    'adjustments_test': ['M', 'm', 'B', 'ADJUST', 'ADJUST2', 'TEST001', 'TEST002'],
    'gift_vouchers': ['gift_0001_10', 'gift_0001_20', 'gift_0001_30', 'gift_0001_40',
                      'gift_0001_50', 'gift_0001_60', 'gift_0001_70', 'gift_0001_80',
                      'gift_0001_90'],
    'samples': ['S'],
    'unclear': ['GIFT', 'C3'] }

In [17]:
for name, codes_list in groups.items():
    mask = df['StockCode'].isin(codes_list) & (df['row_type'] == 'sale')
    df.loc[mask, 'row_type'] = name
    print(name, mask.sum())

charges_fees 4113
adjustments_test 1496
gift_vouchers 100
samples 102
unclear 2


In [18]:
mask = (
    df['StockCode'].str.startswith('DCGS')
    & (df['Description'].isnull() | df['Description'].isin(['ebay', 'update']))
    & (df['row_type'] == 'sale')
)

In [19]:
df.loc[mask, 'row_type'] = 'unclear_dcgs'
print(mask.sum())

33


In [20]:
mask = (df['Invoice'].str.startswith('C'))& (df['row_type']=='sale')

In [21]:
df.loc[mask, 'row_type'] = 'return'

In [22]:
print(mask.sum())

17974


In [23]:
mask = (df['Price']<=0)& (df['row_type']=='sale')

In [24]:
df.loc[mask,'row_type'] = 'zero_or_neg_price'

In [25]:
print(mask.sum())

5940


In [26]:
print((mask & df['Customer ID'].notnull()).sum())

61


In [27]:
df['row_type'].value_counts()

row_type
sale                 1015088
return                 17974
zero_or_neg_price       5940
charges_fees            4113
adjustments_test        1496
samples                  102
gift_vouchers            100
unclear_dcgs              33
unclear                    2
Name: count, dtype: int64

In [28]:
sale = df[df['row_type'] == 'sale']

In [29]:
sale.shape

(1015088, 9)

In [30]:
(sale['Quantity'] <= 0).sum()

np.int64(0)

In [31]:
(sale['Price'] <= 0).sum()

np.int64(0)

In [32]:
(sale['Quantity'] * sale['Price']).sum()

np.float64(19700954.457000002)

Instead of deleting rows, each row gets a `row_type` label, applied in this order so that no row is counted twice:

| row_type | Rows | Share |
|---|---|---|
| sale | 1,015,088 | 97.15% |
| return ('C' invoices) | 17,974 | 1.72% |
| zero_or_neg_price | 5,940 | 0.57% |
| charges_fees | 4,113 | 0.39% |
| adjustments_test | 1,496 | 0.14% |
| samples | 102 | 0.01% |
| gift_vouchers | 100 | 0.01% |
| unclear_dcgs | 33 | <0.01% |
| unclear | 2 | <0.01% |

- **Kept as sales:** `DCGS...`, `PADS` and `SP1002` rows with a product description, because they look like real products with a non-standard code.
- **Zero or negative price:** 61 of the 5,940 rows have a Customer ID. They are excluded from sales but can be identified by this label (the stated reason for the zero price is not in the data).
- **Returns:** kept as a separate group for return-rate and net revenue analysis.
- **Check:** the `sale` group has no rows with Quantity <= 0 or Price <= 0, and its gross revenue (Quantity x Price) is about 19.70 million.

#### Step 5: Description standardization

In [33]:
df['Description'] = df['Description'].str.strip()

In [34]:
df['Description'] = df['Description'].str.replace(r'\s+', ' ', regex=True)

In [35]:
df['Description'] = df['Description'].str.upper()

In [36]:
df['Description'].nunique()

5620

In [37]:
df['Description'].isnull().sum()

np.int64(4275)

In [38]:
sale = df[df['row_type'] == 'sale']

In [39]:
((sale.groupby('StockCode')['Description'].nunique())>1).sum()

np.int64(617)

In [40]:
mapping = (
    sale.groupby(['StockCode', 'Description']).size().reset_index(name='n')
    .sort_values('n', ascending=False)
    .drop_duplicates(subset='StockCode')
    .set_index('StockCode')['Description'])

print(len(mapping))

4896


In [41]:
df['Description'] = df['StockCode'].map(mapping).fillna(df['Description'])

In [42]:
sale = df[df['row_type'] == 'sale']

In [43]:
print((sale.groupby('StockCode')['Description'].nunique() > 1).sum())
print(df['Description'].isnull().sum())
print(df['Description'].nunique())
print(sale['Description'].nunique())

0
406
4748
4695


- **Text cleaning:** Stripped leading and trailing spaces, collapsed repeated spaces and converted to upper case. Distinct descriptions went from 5,698 to 5,620.
- **Standard product name:** Within `sale` rows, 617 StockCodes had more than one Description. Each StockCode now takes its most frequent Description (built from `sale` rows only, so note-like text such as "FOUND" is not used). The count is now 0.
- **Result:** Rows with a missing Description went from 4,275 to 406 (all outside `sale`, in StockCodes that never appear as a sale). `sale` has 4,896 StockCodes but 4,695 distinct descriptions, so some different StockCodes share a Description. **StockCode is the product key, not Description.**

#### Country Standardization

In [44]:
df['Country'].value_counts()

Country
United Kingdom          959983
EIRE                     17689
Germany                  17363
France                   14059
Netherlands               5138
Spain                     3766
Switzerland               3183
Belgium                   3111
Portugal                  2540
Australia                 1891
Channel Islands           1647
Italy                     1509
Sweden                    1364
Norway                    1308
Cyprus                    1176
Finland                   1049
Austria                    938
Denmark                    797
Unspecified                756
Greece                     663
Japan                      565
USA                        535
Poland                     527
United Arab Emirates       500
Israel                     371
Hong Kong                  364
Singapore                  346
Malta                      299
Canada                     228
Iceland                    222
RSA                        169
Lithuania                  154


In [45]:
country = {'EIRE':'Ireland','USA':'United States','RSA':'South Africa'}

In [46]:
df['Country'] = df['Country'].replace(country)

In [47]:
df['Country'].value_counts()

Country
United Kingdom          959983
Ireland                  17689
Germany                  17363
France                   14059
Netherlands               5138
Spain                     3766
Switzerland               3183
Belgium                   3111
Portugal                  2540
Australia                 1891
Channel Islands           1647
Italy                     1509
Sweden                    1364
Norway                    1308
Cyprus                    1176
Finland                   1049
Austria                    938
Denmark                    797
Unspecified                756
Greece                     663
Japan                      565
United States              535
Poland                     527
United Arab Emirates       500
Israel                     371
Hong Kong                  364
Singapore                  346
Malta                      299
Canada                     228
Iceland                    222
South Africa               169
Lithuania                  154


- **Renamed:** EIRE to Ireland, USA to United States, RSA to South Africa. Row counts are unchanged (17,689, 535 and 169) and the number of distinct labels stays at 43.
- **Left as is:** "Korea" (the dataset does not say which country this is), and the non-country labels "Unspecified" (756 rows), "European Community" (61), "West Indies" (54) and "Channel Islands" (1,647). They will be reported as separate categories rather than guessed.
- **Note:** The United Kingdom is about 92% of rows, so country-level comparisons will be dominated by it.

#### Step 6: Sales recalculated

In [48]:
df['Sales'] = df['Quantity']*df['Price']

In [49]:
sale = df[df['row_type'] == 'sale']

In [50]:
sale['Sales'].sum()

np.float64(19700954.457000002)

In [51]:
df['Sales'].sum()

np.float64(18909762.118)

In [52]:
sale['Sales'].min()

0.001

- The original `Sales` column existed only in the first sheet, so it was dropped in Step 2.
- `Sales` is now `Quantity x Price` for every row, kept unrounded because rounding to 2 decimals turns 18 rows with a Price of 0.001 into 0.
- Gross revenue of `sale` rows is about 19.70 million. The whole dataset, including returns and the excluded groups, sums to about 18.91 million.

In [53]:
df['Customer ID'] = df['Customer ID'].astype('Int64')

#### Step 7: Saved clean data

In [54]:
df.to_csv('retail_clean.csv', index=False)

In [55]:
check = pd.read_csv(
    'retail_clean.csv',
    parse_dates=['InvoiceDate'],
    dtype={'StockCode': str, 'Invoice': str, 'Customer ID': 'Int64'}
)

In [56]:
check.shape

(1044848, 10)

In [57]:
check['row_type'].value_counts()

row_type
sale                 1015088
return                 17974
zero_or_neg_price       5940
charges_fees            4113
adjustments_test        1496
samples                  102
gift_vouchers            100
unclear_dcgs              33
unclear                    2
Name: count, dtype: int64

In [58]:
check['Sales'].sum()

np.float64(18909762.118)

In [59]:
check.dtypes

Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID             Int64
Country                object
row_type               object
Sales                 float64
dtype: object

- **File:** `retail_clean.csv`, 1,044,848 rows and 10 columns. It keeps every row, with a `row_type` label, so nothing is lost and every exclusion is traceable.
- **Customer ID:** converted to a nullable integer (5,942 distinct customers, 235,287 rows without an ID).
- **Verified:** re-reading the file gives the same shape, the same `row_type` counts and the same total `Sales`.
- **Use:** analysis runs on `row_type == 'sale'`. Returns are used separately for return-rate and net revenue analysis.

## Cleaning Summary

| Step | Before | After |
|---|---|---|
| Rows (after removing overlap-week copies) | 1,067,371 | 1,044,848 |
| Original `Sales` column (one sheet only) | present in sheet 1 | dropped and recalculated for all rows |
| Rows classed as `sale` | n/a | 1,015,088 (97.15%) |
| Distinct Descriptions | 5,698 | 4,748 |
| Rows without a Description | 4,275 | 406 |
| StockCodes with more than one Description (sale rows) | 617 | 0 |
| Country labels | 43 | 43 (EIRE, USA, RSA renamed) |